In [33]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.animation as animation

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
def leapfrog(u,i,j):
    u[i,j+1] = u[i,j] -  1/40*(u[i+1,j]-u[i-1,j]) + (0.1**2)/8*((u[i+1,j]+u[i,j])*(u[i+1,j]**2+u[i,j]**2)-(u[i,j]+u[i-1,j])*(u[i,j]**2+u[i-1,j]**2))
    return u[i,j+1]
def run():
    def init():
        u = np.zeros((100,100))
        for i in range(100):
            u[i,0] = 3*np.sin(3.2*i)
        u[0,0] = 0
        u[99,0] = 0
        total_steps = 100
        return u, total_steps
    u = []
    u, total_steps = init()
    np.save(output_dir / f"u_{0:04d}.npy", u[:, 0])
    for j in range(total_steps -1):
        u[0, j+1] = 0
        u[99, j+1] = 0
        for i in range(1,99,1):
            leapfrog(u,i,j)
    
        if (j + 1) % 10 == 0:
            np.save(output_dir / f"u_{j+1:04d}.npy", u[:, j+1])
run()


In [36]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

output_dir = Path("output")

files = sorted(output_dir.glob("u_*.npy"))

if not files:
    raise RuntimeError("No output files found.")

print(f"Found {len(files)} snapshots:")
print([f.name for f in files])

# Load snapshots
snapshots = [np.load(f) for f in files]

# Extract timestep numbers
timesteps = [
    int(f.stem.split("_")[1])
    for f in files
]

# Check that the snapshots actually differ
for i, snapshot in enumerate(snapshots):
    print(
        f"{files[i].name}: "
        f"min={snapshot.min():.4e}, "
        f"max={snapshot.max():.4e}, "
        f"norm={np.linalg.norm(snapshot):.4e}"
    )

x = np.arange(len(snapshots[0]))

# Global limits
ymin = min(s.min() for s in snapshots)
ymax = max(s.max() for s in snapshots)

if np.isclose(ymin, ymax):
    ymin -= 1
    ymax += 1

fig, ax = plt.subplots()

line, = ax.plot(x, snapshots[0])

ax.set_xlim(x[0], x[-1])
ax.set_ylim(ymin, ymax)
ax.set_xlabel("Spatial index")
ax.set_ylabel("u")
title = ax.set_title(f"Time step {timesteps[0]}")


def update(frame):
    line.set_ydata(snapshots[frame])
    title.set_text(f"Time step {timesteps[frame]}")
    return line, title


ani = FuncAnimation(
    fig,
    update,
    frames=len(snapshots),
    interval=100,
    blit=True
)

plt.close(fig)

HTML(ani.to_jshtml())

Found 10 snapshots:
['u_0000.npy', 'u_0010.npy', 'u_0020.npy', 'u_0030.npy', 'u_0040.npy', 'u_0050.npy', 'u_0060.npy', 'u_0070.npy', 'u_0080.npy', 'u_0090.npy']
u_0000.npy: min=-2.9999e+00, max=2.9995e+00, norm=2.1838e+01
u_0010.npy: min=-2.9999e+00, max=2.9993e+00, norm=2.1838e+01
u_0020.npy: min=-2.9998e+00, max=2.9991e+00, norm=2.1839e+01
u_0030.npy: min=-2.9997e+00, max=2.9989e+00, norm=2.1840e+01
u_0040.npy: min=-2.9995e+00, max=2.9986e+00, norm=2.1841e+01
u_0050.npy: min=-2.9993e+00, max=2.9983e+00, norm=2.1842e+01
u_0060.npy: min=-2.9993e+00, max=2.9983e+00, norm=2.1843e+01
u_0070.npy: min=-2.9995e+00, max=2.9986e+00, norm=2.1843e+01
u_0080.npy: min=-2.9996e+00, max=2.9988e+00, norm=2.1844e+01
u_0090.npy: min=-2.9997e+00, max=2.9990e+00, norm=2.1845e+01


In [35]:
for f in files:
    data = np.load(f)
    print(f.name, data.min(), data.max(), np.linalg.norm(data))

u_0000.npy -2.999942302489966 2.99948073572562 21.83750896766554
u_0010.npy -2.999873698315077 2.9992994909493835 21.838310572908785
u_0020.npy -2.9997769791542375 2.99908952348045 21.83912908030988
u_0030.npy -2.9996519717248162 2.998850479931329 21.83996345654554
u_0040.npy -2.999498438609681 2.998581938328706 21.840814290462173
u_0050.npy -2.999316076574544 2.998283405200916 21.841682386483104
u_0060.npy -2.9992890599719004 2.9982625349714813 21.842568335803808
u_0070.npy -2.999466806254576 2.9985512433343677 21.84347266652846
u_0080.npy -2.999616686148099 2.998811645215295 21.844395964701693
u_0090.npy -2.9997387976112537 2.9990440073907805 21.845338915913697
